In [1]:
import csv
import pandas as pd
import os
import numpy as np
from datetime import datetime

In [2]:
base_dir = os.getcwd()
mafap_dir = os.path.join(base_dir, "MAFAP")
update_dir = os.path.join(mafap_dir, "Update_2024")
output_dir = os.path.join(update_dir,"Output")

In [3]:
exch = pd.read_csv(os.path.join(update_dir,"Exchange_Rate_2024.csv"), nrows=24)
exch = exch.drop(['Series Name','Series Code'],axis=1)
exch.rename(columns={'Country Code':'Country_Code', 'Country Name':'Country_Label'}, inplace=True)
exch = pd.melt(exch,id_vars=['Country_Label','Country_Code'],var_name='Year', value_name='ER_Official')
exch['Year'] = exch['Year'].apply(lambda x: x.split(' ')[0]).astype(int)
exch = exch.sort_values(['Country_Code','Year'])
exch = exch[exch.Country_Label.notnull()]
exch.Country_Label = exch.Country_Label.str.replace('Kyrgyz Republic','Kyrgyzstan')
print(exch.shape)
exch.head()

(437, 4)


,Country_Label,Country_Code,Year,ER_Official
17,Armenia,ARM,2005,457.686941
41,Armenia,ARM,2006,416.040370
65,Armenia,ARM,2007,342.079116
89,Armenia,ARM,2008,305.969400
113,Armenia,ARM,2009,363.283286


In [4]:
# eth_exch = pd.read_excel(os.path.join(update_dir, "Ethiopia_Exchange_Rate.xlsx"), nrows=2)
# # eth_exch = eth_exch.drop(['Country Name'],axis=1)
# eth_exch.rename(columns={'Country Code':'Country_Code', 'Country Name':'Country_Label'}, inplace=True)
# eth_exch = pd.melt(eth_exch,id_vars=['Country_Label','Country_Code'],var_name='Year', value_name='ER_Official')
# eth_exch['Year'] = eth_exch['Year'].apply(lambda x: x.split(' ')[0]).astype(int)
# eth_exch.head()

In [5]:
# droping WDI data for Ethiopia. Using exchange rate from MAFAP database 
# exch = exch[exch['Country_Code']!='ETH']
# exch = exch.append(eth_exch)
# exch.shape

In [6]:
country_list = exch[['Country_Label','Country_Code']].drop_duplicates()

In [7]:
pmntbycomm = pd.read_excel(os.path.join(update_dir, 'PE_by_COMMODITY_Feb2025update.xlsx'), sheet_name='Sheet1')
pmntbycomm = pmntbycomm[['ISO','Year','Sector_final','Product_final','FAOSTAT_productcode',
                         'FAOSTAT_productname','MPS','Category_code', 'Units', 'actual_expenditures']]
pmntbycomm = pmntbycomm[pmntbycomm['actual_expenditures'].notnull()]

pmntbycomm.actual_expenditures = np.where(pmntbycomm.Units==1000, pmntbycomm.actual_expenditures*1000, 
                                          pmntbycomm.actual_expenditures)


# Ghana-livestock-Not specified, B2 to be 0
pmntbycomm.actual_expenditures = np.where(((pmntbycomm.ISO=='GHA') & (pmntbycomm.Year==2016) & 
                                           (pmntbycomm.Sector_final=='Livestock') & 
                                           (pmntbycomm.Product_final=='Not specified') & 
                                           (pmntbycomm.Category_code=='B2')), 0, 
                                          pmntbycomm.actual_expenditures)


pmntbycomm.drop(['Units'], axis=1, inplace=True)
pmntbycomm.rename(columns={'ISO':'Country_Code', 'Sector_final':'Sector','Product_final':'Product',
                           'FAOSTAT_productcode':'FAO_ProductCode','FAOSTAT_productname':'FAO_ProductName',
                      'actual_expenditures':'Payment', 'Category_code':'Category_Code', 'MPS':'Commodity_Type'}, inplace=True)

pmntbycomm.Sector = np.where(pmntbycomm.Sector=='crops','Crops',np.where(pmntbycomm.Sector=='livestock','Livestock', 
                                np.where(pmntbycomm.Sector=='forestry','Forestry',
                                np.where(pmntbycomm.Sector=='agriculture','Agriculture', pmntbycomm.Sector))))


pmntbycomm = pmntbycomm.replace({'Product':{'maize':'Maize','pig':'Pig', 'oil palm':'Oil palm','shea':'Shea', 
                                            'Eggs':'Egg','cabbage':'Cabbage','yam':"Yam",'Soy':'Soybean',
                                           'Mango ':'Mango','Shea ':'Shea','lettuce':'Lettuce','cassava':'Cassava',
                                           'cocoa':'Cocoa','sesame':'Sesame','goat':'Goat','sheep':'Sheep',
                                           'soybeans,cowpea':'Soybean',' Wheat':'Wheat','Chicken ':'Chicken', 
                                           'fonio':'Fonio','beetroot':'Beet root'}})


# pmntbycomm.FAO_ProductCode = np.where(pmntbycomm.FAO_ProductCode==689,687,pmntbycomm.FAO_ProductCode)

# bgd = pmntbycomm[pmntbycomm.Country_Code=='BGD']
# bgd.Payment = bgd.Payment*1000
# pmntbycomm_wbgd = pmntbycomm[~(pmntbycomm.Country_Code=='BGD')]
# pmntbycomm = pmntbycomm_wbgd.append(bgd)

print(pmntbycomm.shape)
pmntbycomm.head()

(4550, 9)


,Country_Code,Year,Sector,Product,FAO_ProductCode,FAO_ProductName,Commodity_Type,Category_Code,Payment
0,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B1,1.646553e+07
1,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B2,9.075421e+07
2,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B3,3.646539e+06
3,BDI,2005,Livestock,Not specified,NaN,NaN,No,B1,6.136881e+05
4,BDI,2005,Livestock,Not specified,NaN,NaN,No,B2,4.344707e+06


In [8]:
pmntbycomm.Product.unique()

array(['Not specified', 'Rice', 'Coffee', 'Sweet potato', 'Cassava',
       'Potato', 'Banana', 'Maize', 'Coconut', 'all', 'Milk', 'Oil palm',
       'Beans', 'Cotton', 'Pineapple', 'Cashew nut', 'Cane rat', 'Egg',
       'Beef', 'Onion', 'Yam', 'Sesame', 'Sunflower', 'Cow pea',
       'Soybean', 'Bambara beans', 'Mango', 'Millet', 'Sorghum', 'Wheat',
       'Groundnuts', 'Shea', 'Tomato', 'Arabic gum', 'Pig', 'Jute',
       'Sugar', 'Goat', 'Buffalo', 'Carrot', 'Hilsha', 'Apple', 'Bamboo',
       'Sheep', 'Forage', 'Honey', 'Tea', 'Pepper', 'Ginger', 'Chicken',
       'Cocoa', 'Lemon', 'Silk', 'Pyrethrum', 'Fonio', 'Rabbit',
       'Cabbage', nan, 'Beehives', 'Leather', 'Rubber', 'Pawpaw',
       'Turkey', 'Flowers', 'Macadamia', 'Cucumber', 'Lettuce', 'Menthe',
       'Turnip', 'Green pepper', 'Beet root', 'Sisal', 'Tobacco'],
      dtype=object)

In [9]:
pmntcom_newdata = pd.read_excel(os.path.join(update_dir, 'PE_by_commodity_country_format_MAFAP_EECCA_20230110.xlsx'), 
                                sheet_name='Template')
pmntcom_newdata = pmntcom_newdata[['ISO','Year','Sector_final','Product_final','FAOSTAT_productcode',
                         'FAOSTAT_productname','MPS','Category_code', 'Units', 'actual_expenditures']]
pmntcom_newdata = pmntcom_newdata[pmntcom_newdata['actual_expenditures'].notnull()]
pmntcom_newdata.actual_expenditures = np.where(pmntcom_newdata.Units==1000, pmntcom_newdata.actual_expenditures*1000, 
                                               np.where(pmntcom_newdata.Units==1000000, 
                                                        pmntcom_newdata.actual_expenditures*1000000,
                                                        pmntcom_newdata.actual_expenditures))

pmntcom_newdata.drop(['Units'], axis=1, inplace=True)
pmntcom_newdata.rename(columns={'ISO':'Country_Code', 'Sector_final':'Sector','Product_final':'Product',
                           'FAOSTAT_productcode':'FAO_ProductCode','FAOSTAT_productname':'FAO_ProductName',
                      'actual_expenditures':'Payment', 'Category_code':'Category_Code', 'MPS':'Commodity_Type'}, inplace=True)

print(pmntcom_newdata.shape)

(402, 9)


In [10]:
pmntbycomm_excl = pmntbycomm[~pmntbycomm.Sector.isin(['Forestry', 'Fisheries'])]

mushroom =  pmntbycomm[pmntbycomm.Product=='Mushroom']
mushroom.Sector = mushroom.Sector.str.replace('Forestry','Crops')

pmnt_withmush = pmntbycomm_excl.append(mushroom)
pmnt_withmush.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\739231235.py:6: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pmnt_withmush = pmntbycomm_excl.append(mushroom)


(4184, 9)

In [11]:
fishforestry = pmntbycomm[pmntbycomm.Sector.isin(['Forestry', 'Fisheries'])]
fishforestry = fishforestry.groupby(['Country_Code','Year','Category_Code']).sum()[['Payment']].reset_index()
fishforestry.rename(columns={'Category_Code':'Category'}, inplace=True)
fishforestry['Category'] = np.where(fishforestry.Category=='C','F', np.where(fishforestry.Category=='D','C',
                                                                            fishforestry.Category))

In [12]:
product_map = pd.read_excel(os.path.join(mafap_dir, 'Mafap_Commodity_Map.xlsx'))

In [13]:
pmntcom_newdata_f = pmntcom_newdata.merge(product_map, how='left')
print(pmntcom_newdata_f.shape)
pmntcom_newdata_f = pmntcom_newdata_f[~(pmntcom_newdata_f.Commodity_Label.isnull())]
pmntcom_newdata_f.shape

(402, 11)


(402, 11)

In [14]:
commodity_type = pd.read_csv(os.path.join(mafap_dir,'MAFAP_input_file_2024.csv'))
commodity_type = commodity_type[['COUNTRY_CODE','COMMODITY_LABEL','YEAR']].drop_duplicates()
commodity_type.rename(columns={'COUNTRY_CODE':'Country_Code','COMMODITY_LABEL':'Commodity_Label','YEAR':'Year'}, inplace=True)
commodity_type['Commodity_Type'] = 'Yes'
commodity_type.head()

,Country_Code,Commodity_Label,Year,Commodity_Type
0,BGD,Potatoes,2014,Yes
1,BGD,Potatoes,2015,Yes
2,BGD,Potatoes,2016,Yes
3,BGD,Potatoes,2017,Yes
4,BGD,Potatoes,2018,Yes


In [15]:
pmnt_bycom_f = pmnt_withmush.merge(product_map, how='left')
print(pmnt_bycom_f.shape)
pmnt_bycom_f = pmnt_bycom_f[~(pmnt_bycom_f.Commodity_Label.isnull())]
print(pmnt_bycom_f.shape)

(4184, 11)
(4131, 11)


In [16]:
pmntcom_f = pmnt_bycom_f.append(pmntcom_newdata_f)
pmntcom_f.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2369062843.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pmntcom_f = pmnt_bycom_f.append(pmntcom_newdata_f)


(4533, 11)

In [17]:
pmntcom_f = pmntcom_f[['Country_Code', 'Commodity_Label','Commodity_Code','Year','Category_Code','Payment']]

pmntcom_f = pd.merge(pmntcom_f, commodity_type, how='left')

pmntcom_f.Commodity_Type = np.where(pmntcom_f.Commodity_Type.isnull(),'No', pmntcom_f.Commodity_Type)

print(pmntcom_f.shape)

pmntcom_f = pmntcom_f.groupby(['Country_Code','Commodity_Label',
                         'Commodity_Code','Year','Commodity_Type','Category_Code']).sum()[['Payment']].reset_index()
print(pmntcom_f.shape)
pmntcom_f = pmntcom_f.pivot_table(index=['Country_Code','Commodity_Label',
                                               'Commodity_Code','Commodity_Type','Year'], columns='Category_Code', 
                    values=['Payment'])

print(pmntcom_f.shape)

pmntcom_f = pmntcom_f.sort_index(axis=1, level=1)
print(pmntcom_f.shape)

pmntcom_f.columns = [f'{y}' for x,y in pmntcom_f.columns]
pmntcom_f = pmntcom_f.reset_index()
pmntcom_f = pmntcom_f.merge(country_list, how='left')
print(pmntcom_f.shape)

# pmntcom_f['A'] = np.nan
pmntcom_f['E'] = np.nan
pmntcom_f['F'] = np.nan
pmntcom_f['G'] = np.nan

pmntcom_f = pmntcom_f[['Country_Label','Country_Code','Commodity_Label','Commodity_Code', 'Commodity_Type',
                             'Year','A','B1','B2','B3','C','D','E','F','G']]

# Following two commodities to be tagged as non-MPS commodity 
pmntcom_f.Commodity_Type = np.where((pmntcom_f.Country_Label=='Burkina Faso') & 
                                       (pmntcom_f.Commodity_Label=='Onions'), 'No', pmntcom_f.Commodity_Type)

pmntcom_f.Commodity_Type = np.where((pmntcom_f.Country_Label=='Ghana') & 
                                       (pmntcom_f.Commodity_Label=='Cassava'), 'No',pmntcom_f.Commodity_Type)


print(pmntcom_f.shape)
pmntcom_f.head()

(4533, 7)
(3845, 7)
(2174, 6)
(2174, 6)
(2174, 12)
(2174, 15)


,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A,B1,B2,B3,C,D,E,F,G
0,Armenia,ARM,Non-allocated crops,9991.0,No,2006,NaN,3.710200e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Armenia,ARM,Non-allocated crops,9991.0,No,2007,NaN,5.784722e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Armenia,ARM,Non-allocated crops,9991.0,No,2008,NaN,6.583030e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Armenia,ARM,Non-allocated crops,9991.0,No,2009,NaN,6.852542e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Armenia,ARM,Non-allocated crops,9991.0,No,2010,NaN,4.756000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
nonmps = pmntcom_f[pmntcom_f.Commodity_Label.isin(['Non-MPS other livestock', 'Non-MPS other crops'])]
nonmps['Commodity_Label'] = nonmps.Commodity_Label + ' - ' + nonmps['Country_Label']

nmpsmap = {'Non-MPS other livestock - Burundi':99940,
       'Non-MPS other livestock - Benin':99931,
       'Non-MPS other livestock - Burkina Faso':99932,
       'Non-MPS other livestock - Ethiopia':99933,
       'Non-MPS other livestock - Kenya':99934,
       'Non-MPS other livestock - Mali':99935,
       'Non-MPS other livestock - Mozambique':99936,
       'Non-MPS other livestock - Malawi':99937, 
        'Non-MPS other crops - Rwanda':99941,       
       'Non-MPS other livestock - Senegal':99939,
          'Non-MPS other livestock - Rwanda':99938, 
          'Non-MPS other livestock - Azerbaijan':99942, 
          'Non-MPS other livestock - Nigeria':99943} 

nonmps.Commodity_Type = np.where(nonmps.Commodity_Type=='Yes', 'No', nonmps.Commodity_Type)

nonmps['Commodity_Code'] = nonmps.Commodity_Label.map(nmpsmap)
nonmps.head()


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\4096813935.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nonmps['Commodity_Label'] = nonmps.Commodity_Label + ' - ' + nonmps['Country_Label']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\4096813935.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nonmps.Commodity_Type = np.where(nonmps.Commodity_Type=='Yes', 'No', nonmps.Commodity_Type)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\4096813935.py:20: SettingWithCopyWarning: 
A value is t

,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A,B1,B2,B3,C,D,E,F,G
35,Azerbaijan,AZE,Non-MPS other livestock - Azerbaijan,99942,No,2017,1199000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,Azerbaijan,AZE,Non-MPS other livestock - Azerbaijan,99942,No,2018,2490000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,Benin,BEN,Non-MPS other livestock - Benin,99931,No,2011,NaN,4.549249e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233,Benin,BEN,Non-MPS other livestock - Benin,99931,No,2012,NaN,1.838589e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN
234,Benin,BEN,Non-MPS other livestock - Benin,99931,No,2013,NaN,2.668672e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
not_nonmps = pmntcom_f[~(pmntcom_f.Commodity_Label.isin(['Non-MPS other livestock', 'Non-MPS other crops']))]
print(not_nonmps.shape)
mafap_relab = not_nonmps.append(nonmps)
print(mafap_relab.shape)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\1554685287.py:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mafap_relab = not_nonmps.append(nonmps)


(2091, 15)
(2174, 15)


In [20]:
pmntot = pd.read_excel(os.path.join(update_dir, 'Global_PE_DEC2024update.xlsx'), sheet_name='Common')
pmntot.shape

(6306, 52)

In [21]:
pmntot_new = pd.read_excel(os.path.join(update_dir,'Public expenditure dataset_MAFAP_EECCA 27March2023.xlsx'), 
                              sheet_name='PE_Oct2022')
pmntot_new.Country = pmntot_new.Country.str.replace('Kyrgystan','Kyrgyzstan')
pmntot_new.Country = pmntot_new.Country.str.replace('Republic of Moldova','Moldova')
pmntot_new.shape

(74, 51)

In [22]:
pmnttotal = pmntot.append(pmntot_new)
print(pmnttotal.shape)
pmnttotal = pmnttotal[pmnttotal['Amounts expressed in']=='LCU, nominal']
pmnttotal = pmnttotal[pmnttotal['Budgeted/actual']=='Actual']
pmnttotal = pmnttotal[pmnttotal['Donor/national/total']=='Total']
pmnttotal = pmnttotal[['Country', 'Year', 
       'A. Production subsidies based on outputs', 'B. Input subsidies',
       'B1. Variable inputs',
       'B2. Capital (including on-farm irrigation and infrastructure)',
       'B3. On-farm services', 'C. Income support',
       'D. Other payments to producers', 'I.1.2. Payments to consumers',
       'E. Food aid', 'F. Cash transfers', 'G. School feeding programmes',
       'H. Other payments to consumers', 'I.1.3. Payments to input suppliers',
       'I.1.4. Payments to processors', 'I.1.5. Payments to traders',
       'I.1.6. Payments to transporters',
       'I.2. General support to the food and agriculture sector',
       'I. Agricultural research', 'J. Technical assistance', 'K. Training',
       'L. Extension/technology transfer', 'M. Inspection',
       'N. Agricultural infrastructure', 'N1. Feeder roads',
       'N2. Off-farm irrigation', 'N3. Other off-farm infrastructure',
       'O. Storage/public stockholding', 'P. Marketing',
       'Q. Other general support to the food and agriculture sector']]

pmnttotal.rename(columns={'Country':'Country_Label','A. Production subsidies based on outputs':'A', 'B. Input subsidies':'B',
       'B1. Variable inputs':'B1',
       'B2. Capital (including on-farm irrigation and infrastructure)':'B2',
       'B3. On-farm services':'B3', 'C. Income support':'C',
       'D. Other payments to producers':'D', 'I.1.2. Payments to consumers':'I_1_2',
       'E. Food aid':'E', 'F. Cash transfers':'F', 'G. School feeding programmes':'G',
       'H. Other payments to consumers':'H', 'I.1.3. Payments to input suppliers':'I_1_3',
       'I.1.4. Payments to processors':'I_1_4', 'I.1.5. Payments to traders':'I_1_5',
       'I.1.6. Payments to transporters':'I_1_6',
       'I.2. General support to the food and agriculture sector':'I_2',
       'I. Agricultural research':'I', 'J. Technical assistance':'J', 'K. Training':'K',
       'L. Extension/technology transfer':'L', 'M. Inspection':'M',
       'N. Agricultural infrastructure':'N', 'N1. Feeder roads':'N1',
       'N2. Off-farm irrigation':'N2', 'N3. Other off-farm infrastructure':'N3',
       'O. Storage/public stockholding':'O', 'P. Marketing':'P',
       'Q. Other general support to the food and agriculture sector':'Q'}, inplace=True)

pmnttotal = pmnttotal.merge(country_list, how='left')

pmnttotal.head()

(6380, 53)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\3105610434.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pmnttotal = pmntot.append(pmntot_new)


,Country_Label,Year,A,B,B1,B2,B3,C,D,I_1_2,...,L,M,N,N1,N2,N3,O,P,Q,Country_Code
0,Bangladesh,2019,5.281995e+10,8.351490e+10,8.134767e+10,1.657640e+09,5.095942e+08,0.0,4.503829e+09,3.284851e+10,...,2.275211e+10,6.968865e+09,2.070806e+10,1.053640e+08,1.844027e+10,2.162430e+09,4.900815e+10,9.097426e+09,1.168022e+10,BGD
1,Bangladesh,2020,5.009899e+10,7.744878e+10,7.591155e+10,1.026897e+09,5.103358e+08,0.0,3.779166e+09,4.134720e+10,...,2.484208e+10,7.277482e+09,1.074509e+10,3.850000e+07,9.645166e+09,1.061425e+09,3.832654e+10,9.039647e+09,1.270610e+10,BGD
2,Bangladesh,2021,3.467833e+10,8.897532e+10,8.431262e+10,4.495971e+09,1.667303e+08,0.0,4.476901e+09,3.935117e+10,...,2.489364e+10,8.053276e+09,1.535680e+10,2.450000e+08,1.054995e+10,4.561850e+09,7.544852e+10,9.759625e+09,1.636404e+10,BGD
3,Bangladesh,2022,4.591588e+10,1.097451e+11,1.042516e+11,5.308933e+09,1.845991e+08,0.0,8.091300e+07,3.591737e+09,...,2.110827e+10,9.009707e+09,9.579678e+09,1.704412e+08,8.615015e+09,7.942223e+08,7.603819e+10,7.975189e+09,1.361951e+10,BGD
4,Benin,2008,0.000000e+00,4.326807e+09,1.468708e+09,2.699549e+09,1.585510e+08,309250851.0,3.247761e+07,8.610692e+08,...,3.330053e+09,2.478943e+09,5.341443e+09,2.810768e+09,1.060565e+09,1.470111e+09,1.055765e+09,3.197457e+09,1.689156e+10,BEN


In [23]:
pmnt_aefg = pmnttotal[['Country_Label','Country_Code', 'Year', 'A']]
pmnt_aefg = pmnt_aefg[~(pmnt_aefg.Country_Label.isin(['Azerbaijan','Georgia','Bangladesh', 'Tanzania']))]
print(pmnt_aefg.shape)
pmnt_aefg['Commodity_Label'] = 'Unallocated'
pmnt_aefg['Commodity_Code'] = 9999
pmnt_aefg['Commodity_Type'] = np.nan

pmnt_aefg['B1'] = np.nan
pmnt_aefg['B2'] = np.nan
pmnt_aefg['B3'] = np.nan
pmnt_aefg['C'] = np.nan
pmnt_aefg['D'] = np.nan
pmnt_aefg['E'] = np.nan
pmnt_aefg['F'] = np.nan
pmnt_aefg['G'] = np.nan

pmnt_aefg = pmnt_aefg[['Country_Label','Country_Code','Commodity_Label','Commodity_Code', 'Commodity_Type',
                             'Year','A','B1','B2','B3','C','D','E','F','G']]
print(pmnt_aefg.shape)
pmnt_aefg.head()

(255, 4)
(255, 15)


,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A,B1,B2,B3,C,D,E,F,G
4,Benin,BEN,Unallocated,9999,NaN,2008,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Benin,BEN,Unallocated,9999,NaN,2009,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Benin,BEN,Unallocated,9999,NaN,2010,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Benin,BEN,Unallocated,9999,NaN,2011,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Benin,BEN,Unallocated,9999,NaN,2012,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# tza_web = pmnt_aefg[pmnt_aefg.Country_Label=='Tanzania']
# tza_web.A = 0
# print(tza_web.shape)
# print(pmnt_aefg.shape)

# pmnt_aefg_wtza = pmnt_aefg[~(pmnt_aefg.Country_Label=='Tanzania')]
# print(pmnt_aefg_wtza.shape)

# pmnt_web_f = pmnt_aefg_wtza.append(tza_web)

In [25]:
# tza = pmnttotal[['Country_Label','Country_Code','Year','C']][pmnttotal.Country_Label=='Tanzania']
# tza.rename(columns={'C':'C_web'}, inplace=True)
# tza

In [26]:
mafap_relab.shape

(2174, 15)

In [27]:
tza_com = mafap_relab[(mafap_relab.Country_Label=='Tanzania')]
print(tza_com.shape)

tza_unalloc = tza_com[tza_com.Commodity_Label.isin(['Non-allocated agriculture','Non-allocated livestock'])]

tza_unalloc.C = 0
print(tza_unalloc.shape)

tza_oth = tza_com[~(tza_com.Commodity_Label.isin(['Non-allocated agriculture','Non-allocated livestock']))]

tza_com = tza_oth.append(tza_unalloc).reset_index(drop=True)
print(tza_com.shape)
print(pmnt_bycom_f.shape)
pmnt_bycom_f.head()

(49, 15)
(7, 15)
(49, 15)
(4131, 11)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2918561321.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tza_unalloc.C = 0
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2918561321.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  tza_com = tza_oth.append(tza_unalloc).reset_index(drop=True)


,Country_Code,Year,Sector,Product,FAO_ProductCode,FAO_ProductName,Commodity_Type,Category_Code,Payment,Commodity_Label,Commodity_Code
0,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B1,1.646553e+07,Unallocated,9999.0
1,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B2,9.075421e+07,Unallocated,9999.0
2,BDI,2005,Agriculture,Not specified,NaN,NaN,No,B3,3.646539e+06,Unallocated,9999.0
3,BDI,2005,Livestock,Not specified,NaN,NaN,No,B1,6.136881e+05,Non-allocated livestock,9992.0
4,BDI,2005,Livestock,Not specified,NaN,NaN,No,B2,4.344707e+06,Non-allocated livestock,9992.0


In [28]:
without_tza = mafap_relab[~(mafap_relab.Country_Label=='Tanzania')]
print(without_tza.shape)

pmnt_com_final = without_tza.append(tza_com).reset_index(drop=True)
print(pmnt_com_final.shape)
pmnt_com_final.head()

(2125, 15)
(2174, 15)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\3324087317.py:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pmnt_com_final = without_tza.append(tza_com).reset_index(drop=True)


,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A,B1,B2,B3,C,D,E,F,G
0,Armenia,ARM,Non-allocated crops,9991.0,No,2006,NaN,3.710200e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Armenia,ARM,Non-allocated crops,9991.0,No,2007,NaN,5.784722e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Armenia,ARM,Non-allocated crops,9991.0,No,2008,NaN,6.583030e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Armenia,ARM,Non-allocated crops,9991.0,No,2009,NaN,6.852542e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Armenia,ARM,Non-allocated crops,9991.0,No,2010,NaN,4.756000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
pmnt_all = pmnttotal[['Country_Label','Country_Code','Year','A','B1','B2','B3','C','D']]
pmnt_all.rename(columns={'A':'A2','C':'F','D':'C'}, inplace=True)
pmnt_all['A1'] = np.nan
pmnt_all['D'] = np.nan
pmnt_all['E'] = np.nan
pmnt_all['G'] = np.nan

pmnt_all = pmnt_all[['Country_Label','Country_Code','Year','A1','A2', 'B1','B2','B3', 'C','D','E','F','G']]

# pmnt_all.to_csv(os.path.join(mafap_dir, 'MAFAP_Payment_Web.csv'), index=False)

pmnt_all = pd.melt(pmnt_all, id_vars=['Country_Label','Country_Code','Year'], var_name='Category', value_name='Payment_web')
pmnt_all = pmnt_all.sort_values(['Country_Label','Country_Code','Year','Category'])

# bgd_total = pmnt_all[pmnt_all.Country_Code=='BGD']
# bgd_total.Payment_web = bgd_total.Payment_web*1000

# pmnt_all_wbgd = pmnt_all[~(pmnt_all.Country_Code=='BGD')]
# pmnt_all = pmnt_all_wbgd.append(bgd_total)
pmnt_all.head()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2441847513.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pmnt_all.rename(columns={'A':'A2','C':'F','D':'C'}, inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2441847513.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pmnt_all['A1'] = np.nan
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2441847513.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

,Country_Label,Country_Code,Year,Category,Payment_web
220,Armenia,ARM,2005,A1,NaN
514,Armenia,ARM,2005,A2,NaN
808,Armenia,ARM,2005,B1,0.000000e+00
1102,Armenia,ARM,2005,B2,1.369900e+09
1396,Armenia,ARM,2005,B3,NaN


In [30]:
pmnt_df = pmnt_com_final.append(pmnt_aefg)
print(pmnt_df.shape)
pmnt_df = pmnt_df[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type','Year','A', 
                  'B1','B2','B3', 'C','D']]
pmnt_df.rename(columns={'A':'A2', 'C':'F', 'D':'C'}, inplace=True)
pmnt_df['D'] = np.nan
pmnt_df['E'] = np.nan
pmnt_df['G'] = np.nan

mafap_pmnt_df = pmnt_df[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type','Year','A2', 
                  'B1','B2','B3', 'C','D','E','F','G']]

mafap_pmnt_df['B'] = mafap_pmnt_df['B1'].fillna(0) + mafap_pmnt_df['B2'].fillna(0) + mafap_pmnt_df['B3'].fillna(0)

mafap_pmnt_df = mafap_pmnt_df[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type','Year','A2', 
                  'B','B1','B2','B3','C','D','E','F','G']]


(2429, 15)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2428166250.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  pmnt_df = pmnt_com_final.append(pmnt_aefg)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2428166250.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mafap_pmnt_df['B'] = mafap_pmnt_df['B1'].fillna(0) + mafap_pmnt_df['B2'].fillna(0) + mafap_pmnt_df['B3'].fillna(0)


In [31]:
# pmnt_aefg_wtza.Country_Label.unique()

In [32]:
pmnt_agg = pmnt_df.groupby(['Country_Label','Country_Code','Year']).sum()[['A2','B1','B2','B3','C','D','E','F','G']].reset_index()

pmnt_agg = pd.melt(pmnt_agg, id_vars=['Country_Label','Country_Code','Year'], var_name='Category', value_name='Payment_sum')
pmnt_agg = pmnt_agg.sort_values(['Country_Label','Country_Code','Year','Category'])

pmnt_agg.shape

(2376, 5)

In [33]:
pmnt_ch = pmnt_all.merge(pmnt_agg)
pmnt_ch['Ratio'] = pmnt_ch['Payment_sum']/pmnt_ch['Payment_web']

pmnt_ch.to_excel(os.path.join(output_dir, 'MAFAP_Data_Add_Up.xlsx'), index=False)

print(pmnt_ch.shape)
print(pmnt_ch[pmnt_ch.Ratio>1].shape)
print(pmnt_ch[pmnt_ch.Ratio>2].shape)
print(pmnt_ch[pmnt_ch.Ratio==1].shape)
print(pmnt_ch[pmnt_ch.Ratio<1].shape)
print(pmnt_ch[pmnt_ch.Ratio.isnull()].shape)


(2304, 7)
(78, 7)
(17, 7)
(376, 7)
(390, 7)
(1460, 7)


In [34]:
pmnt_remain = pmnt_ch.merge(fishforestry, how='left')
pmnt_remain = pmnt_remain[pmnt_remain.Ratio<1]
pmnt_remain['Payment_unallocated'] = pmnt_remain.Payment_web.fillna(0)-pmnt_remain.Payment_sum.fillna(0)-pmnt_remain.Payment.fillna(0)
# negligible difference filtered out
pmnt_remain = pmnt_remain[pmnt_remain.Payment_unallocated>10]
pmnt_remain['Commodity_Label'] = 'Unallocated'
pmnt_remain['Commodity_Code'] = 9999
pmnt_remain = pmnt_remain[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Year',
                           'Category', 'Payment_unallocated']]

pmnt_remain = pmnt_remain.pivot_table(index=['Country_Label','Country_Code','Commodity_Label',
                                               'Commodity_Code','Year'], columns='Category', 
                    values=['Payment_unallocated'])

pmnt_remain = pmnt_remain.sort_index(axis=1, level=1)

pmnt_remain.columns = [f'{y}' for x,y in pmnt_remain.columns]
pmnt_remain = pmnt_remain.reset_index()
pmnt_remain['Commodity_Type'] = 'No'
pmnt_remain['A2'] = np.nan
pmnt_remain['D'] = np.nan
pmnt_remain['E'] = np.nan
pmnt_remain['G'] = np.nan

pmnt_remain['B'] = pmnt_remain['B1'].fillna(0) + pmnt_remain['B2'].fillna(0) + pmnt_remain['B3'].fillna(0)

pmnt_remain = pmnt_remain[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type','Year','A2', 
                 'B', 'B1','B2','B3', 'C','D','E','F','G']]

final_mafapdata = mafap_pmnt_df.append(pmnt_remain)

final_mafapdata.Commodity_Type = np.where(final_mafapdata.Commodity_Label=='Unallocated','No', final_mafapdata.Commodity_Type)

final_mafapdata = final_mafapdata[~(final_mafapdata.Country_Label.isin(['Mauritania','Seychelles','Niger','Sierra Leone']))]

final_mafapdata.Commodity_Code = final_mafapdata.Commodity_Code.astype(str)

final_mafapdata = final_mafapdata[~(final_mafapdata.Country_Code=='ZWE')]


final_mafapdata.to_csv(os.path.join(output_dir, 'MAFAP_Payment_Data.csv'), index=False)
final_mafapdata.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_33524\2257384903.py:30: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  final_mafapdata = mafap_pmnt_df.append(pmnt_remain)


(2454, 16)

In [35]:
# mafap_nrp = pd.read_csv(os.path.join(mafap_dir, 'MAFAP_input_file.csv'))
# mafap_nrp = mafap_nrp[['COUNTRY_LABEL','COUNTRY_CODE','COMMODITY_LABEL','COMMODITY_CODE','YEAR','PRODQ','REFP']]
# mafap_nrp['VP_REFP'] = mafap_nrp['PRODQ']*mafap_nrp['REFP']
# mafap_nrp.rename(columns={'COUNTRY_LABEL':'Country_Label','COUNTRY_CODE':'Country_Code',
#                           'COMMODITY_LABEL':'Commodity_Label','COMMODITY_CODE':'Commodity_Code','YEAR':'Year'}, inplace=True)
# mafap_nrp = mafap_nrp[['Country_Label','Country_Code','Commodity_Label','Commodity_Code','Year', 'VP_REFP']]